In [117]:
#setup
import napari
import numpy as np
from skimage import io, util, measure

import os
import re
from itertools import combinations#for organelle overlaps

viewer = napari.Viewer()

In [20]:
#organelle codes
organelles = ['nuclei', 'Golgi', 'peroxisomes', 'ER', 'mitochondria', 'lysosomes', 'other']
organelles_short = {'nuclei':'n', 'Golgi':'g', 'peroxisomes':'p', 'ER':'e', 'mitochondria':'m', 'lysosomes':'l', 'other':'o'}#for pixel overlaps; must be unique

In [21]:
#***USER DEFAULTS***
#change these values to set your default cell/organelle suffixes
default_cells = "_cell_masks"
default_org = dict()
default_org["nuclei"] = "_nuclei"
default_org["Golgi"] = "_golgi"
default_org["peroxisomes"] = "_perox"
default_org["ER"] = "_ER"
default_org["mitochondria"] = "_mito"
default_org["lysosomes"] = "_lyso"
default_org["other"] = "_other"
default_checkboxes = [False, True, True, True, True, False, True]#in same organelle order as above (cell boundaries are mandatory)
other_name = "Other"
#*********************

datadir = 'C:\\Users\\fiona\\Desktop\\NAPARI_WORKSHOP_DATA\\ONE_SET'


In [22]:
f_ext = set()
filenames = []

for obj in os.listdir(datadir):
	if os.path.isfile(datadir + "/" + obj):
		filenames.append(obj)
		f_ext.add(re.search(r"\.([A-Za-z0-9]+)$", obj).group(1))

#create analysis folder for output
if not os.path.exists(datadir + "/analysis/"):
	os.mkdir(datadir + "/analysis/")


In [50]:
#SPOOFING USER INPUT - integrate here
ext = 'tif'
cell_bool = True
# if cell_bool != True:
#     raise Exception("Cell boundaries are required for processing organelle segmentations.")
cell_regex = default_cells

default_check = dict()
default_check["nuclei"] = False
default_check["Golgi"] = True
default_check["peroxisomes"] = True
default_check["ER"] = True
default_check["mitochondria"] = True
default_check["lysosomes"] = False
default_check["other"] = True

org_bool = dict()
org_regex = dict()#not actually regex - just using str.replace for now
for o in organelles:
    org_bool[o] = default_check[o]
    org_regex[o] = default_org[o]
other_name = 'bacteria'
contacts_bool = True
nuclei_bool = False

nuclei
False
_nuclei
Golgi
True
_golgi
peroxisomes
True
_perox
ER
True
_ER
mitochondria
True
_mito
lysosomes
False
_lyso
other
True
_other


In [51]:
org_regex = {k:v for (k,v), b in zip(org_regex.items(), list(org_bool.values())) if b == True}

{'nuclei': '_nuclei', 'Golgi': '_golgi', 'peroxisomes': '_perox', 'ER': '_ER', 'mitochondria': '_mito', 'lysosomes': '_lyso', 'other': '_other'}
{'Golgi': '_golgi', 'peroxisomes': '_perox', 'ER': '_ER', 'mitochondria': '_mito', 'other': '_other'}


In [52]:
organelles_selected = []
for o in organelles:
    if org_bool[o] == True:
        organelles_selected.append(o)
        
if org_bool["other"] == True:#TODO - make this a bit neater
    org_regex[other_name] = org_regex["other"]
    del org_regex["other"]
    organelles_short[other_name] = organelles_short["other"]
    del organelles_short["other"]
    organelles_selected = [other_name if o == "other" else o for o in organelles_selected]#replace with custom organelle name

nuclei
Golgi
TRUE
peroxisomes
TRUE
ER
TRUE
mitochondria
TRUE
lysosomes
other
TRUE


In [54]:
#ROI groups - convert to labels?
ROI_groups = dict(zip(organelles_selected, list(range(2, len(organelles_selected) + 2, 1))))
ROI_groups["cells"] = 1


In [55]:
#contact groups and ROI groups
if (contacts_bool == True and len(organelles_selected) >= 2):
    contact_combos = []
    for n in range(2, len(organelles_selected)+1, 1):
        contact_combos = contact_combos + list(combinations(organelles_selected, n))
    contact_groups_keys = []
    for g in contact_combos:
        contact_groups_keys.append(''.join([organelles_short.get(key) for key in g]))
    contact_groups = dict(zip(contact_groups_keys, contact_combos))#
    
    contact_ROI_groups = dict(zip(contact_groups.keys(), range(max(ROI_groups.values()) + 1, max(ROI_groups.values()) + 1 + len(contact_groups), 1)))	


[('Golgi', 'peroxisomes'), ('Golgi', 'ER'), ('Golgi', 'mitochondria'), ('Golgi', 'bacteria'), ('peroxisomes', 'ER'), ('peroxisomes', 'mitochondria'), ('peroxisomes', 'bacteria'), ('ER', 'mitochondria'), ('ER', 'bacteria'), ('mitochondria', 'bacteria'), ('Golgi', 'peroxisomes', 'ER'), ('Golgi', 'peroxisomes', 'mitochondria'), ('Golgi', 'peroxisomes', 'bacteria'), ('Golgi', 'ER', 'mitochondria'), ('Golgi', 'ER', 'bacteria'), ('Golgi', 'mitochondria', 'bacteria'), ('peroxisomes', 'ER', 'mitochondria'), ('peroxisomes', 'ER', 'bacteria'), ('peroxisomes', 'mitochondria', 'bacteria'), ('ER', 'mitochondria', 'bacteria'), ('Golgi', 'peroxisomes', 'ER', 'mitochondria'), ('Golgi', 'peroxisomes', 'ER', 'bacteria'), ('Golgi', 'peroxisomes', 'mitochondria', 'bacteria'), ('Golgi', 'ER', 'mitochondria', 'bacteria'), ('peroxisomes', 'ER', 'mitochondria', 'bacteria'), ('Golgi', 'peroxisomes', 'ER', 'mitochondria', 'bacteria')]
('Golgi', 'peroxisomes')
('Golgi', 'ER')
('Golgi', 'mitochondria')
('Golgi', 

In [58]:
filenames_filtered = [f for f in filenames if re.search(ext + "$", f)]#filter to files with the correct extension

#find conditions from filenames
conditions = set()
filenames_key = dict()
for f in filenames_filtered:
	g = f #previously g = f.replace("." + ext, "")
	for o in org_regex.values():
		g = g.replace(o + "." + ext, "")#not actually regex matching - probably best for now
	g = g.replace(cell_regex + "." + ext, "")
	if not g.endswith("." + ext):#trying to exclude file names that didn't match any of the cell/organelle values
		conditions.add(g)
	if g in filenames_key.keys():
		filenames_key[g].append(f)
	else:
		filenames_key[g] = [f]

['1dpi_infected_U2OS_1.tif', '1dpi_infected_U2OS_1_cell_masks.tif', '1dpi_infected_U2OS_1_ER.tif', '1dpi_infected_U2OS_1_golgi.tif', '1dpi_infected_U2OS_1_mito.tif', '1dpi_infected_U2OS_1_other.tif', '1dpi_infected_U2OS_1_perox.tif', 'sample_segmentations.zip']
{'1dpi_infected_U2OS_1'}


In [61]:
##MAIN LOOP
progress = 0
for c in conditions:
    #SPLITTING UP FOR NOW - MANUALLY SETTING C
    print(c)#placeholder

1dpi_infected_U2OS_1


In [63]:
#TEMPORARY
c = '1dpi_infected_U2OS_1'

In [64]:
print("Starting condition " + c)
progress += 1
#open and rename images
c_filenames = filenames_key[c]#find filenames matching condition
c_cell_image = str()
imp_key = {}
for cf in c_filenames:
    imp = io.imread(os.path.join(datadir, cf))#read in file
    try:
        organelle = [k for k,v in org_regex.items() if v in cf][0]
        imp_key[organelle] = imp
    except:#index out of range - no match in organelles
        if cell_regex in cf:
            c_cell_image = cf

Starting condition 1dpi_infected_U2OS_1


In [67]:
images_to_stack = dict()
#ORGANELLE THRESHOLDING
for org in organelles_selected:
    org_img = imp_key[org]
    org_img = org_img > 0#CHANGED
    images_to_stack[org] = (org_img)
    
#ORGANELLE OVERLAPS
for combo in contact_groups.keys():
    combo_res = imp_key[contact_groups[combo][0]]
    combo_i = 1
    while(combo_i < len(contact_groups[combo])):
        combo_res = np.logical_and(combo_res, imp_key[contact_groups[combo][combo_i]])
        combo_i += 1
    images_to_stack[combo] = (combo_res)
    

In [80]:
#STACK
orgstack = io.concatenate_images(images_to_stack.values())
    
#LOAD CELL IMAGE
cells = io.imread(os.path.join(datadir, c_cell_image))

#Check max value
cells_max = int(cells.max()) #converting double to int - beware of rounding bugs
print(cells_max)

3


In [ ]:
#create dict for saving ROIs (aligned to original image)
cells_ROIs_orig = dict()
pooled_ROIs_per_cell = dict()

In [122]:
#Per step (cell) loop:
cells_rps = dict()
viewer.add_labels(cells)
for i in range(1, cells_max + 1, 1):
    print("Processing cell " + str(i) + "...")
    cell_id = "cell_" + str(i)
    #duplicate cell image
    cells_copy = cells
    #step threshold
    cells_copy = (cells_copy == i)*cells_copy
   
    if int(cells_copy.sum()) == 0:#No cell for this value - excluded, deleted, etc.
        continue
   
    cell_rps = measure.regionprops(cells_copy)
    print(cell_rps[0])#CHECK HOW THIS HANDLES DISCONTINUOUS ETC.
    cells_rps[cell_id] = cell_rps[0]
    cell_bbox = cell_rps[0].bbox
    org_crop = np.logical_and(orgstack, cells_copy>0)#clear outside of thresholded cell boundary
    cell_crop = util.crop(org_crop, ((0,0), (cell_bbox[0], org_crop.shape[1]-cell_bbox[2]), (cell_bbox[1], org_crop.shape[2]-cell_bbox[3])))

    
    viewer.add_labels(cell_crop, name = cell_id, translate = (0, cell_bbox[0], cell_bbox[1]))#translate not working

    io.imsave(datadir + "/analysis/" + c + "_" + cell_id + "_stack.tif", cell_crop)



Processing cell 1...


C:\Users\fiona\AppData\Local\Temp\ipykernel_18856\1183972332.py:42: UserWarning: C:\Users\fiona\Desktop\NAPARI_WORKSHOP_DATA\ONE_SET/analysis/1dpi_infected_U2OS_1_cell_1_stack.tif is a boolean image: setting True to 255 and False to 0. To silence this warning, please convert the image using img_as_ubyte.
  io.imsave(datadir + "/analysis/" + c + "_" + cell_id + "_stack.tif", cell_crop)


Processing cell 2...


C:\Users\fiona\AppData\Local\Temp\ipykernel_18856\1183972332.py:42: UserWarning: C:\Users\fiona\Desktop\NAPARI_WORKSHOP_DATA\ONE_SET/analysis/1dpi_infected_U2OS_1_cell_2_stack.tif is a boolean image: setting True to 255 and False to 0. To silence this warning, please convert the image using img_as_ubyte.
  io.imsave(datadir + "/analysis/" + c + "_" + cell_id + "_stack.tif", cell_crop)


Processing cell 3...


C:\Users\fiona\AppData\Local\Temp\ipykernel_18856\1183972332.py:42: UserWarning: C:\Users\fiona\Desktop\NAPARI_WORKSHOP_DATA\ONE_SET/analysis/1dpi_infected_U2OS_1_cell_3_stack.tif is a boolean image: setting True to 255 and False to 0. To silence this warning, please convert the image using img_as_ubyte.
  io.imsave(datadir + "/analysis/" + c + "_" + cell_id + "_stack.tif", cell_crop)


<Labels layer 'cells' at 0x268140c1970>